In [83]:
import pandas as pd
import re
import emoji
from pathlib import Path

In [84]:
RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [85]:
df_talkmap = pd.read_csv(RAW_DIR / "telecom_100k.csv")
df_comcast = pd.read_csv(RAW_DIR / "Comcast.csv")
df_bitext = pd.read_csv(RAW_DIR / "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")

In [ ]:
def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")


In [ ]:
def remove_repeated_tokens(text):
    """
    Collapse repeated words or symbols into a single occurrence.
    Example:
    'to to to' -> 'to'
    '%. %. %.' -> '%.'
    """
    return re.sub(r'\b(\S+)(\s+\1)+\b', r'\1', text)


In [ ]:
def remove_parenthetical_noise(text):
    """
    Removes parenthetical expressions that describe actions, emotions,
    or emoji descriptions, e.g. (To self), (Thumbs up and a smile)
    """
    return re.sub(r"\([^)]*\)", "", text)


In [88]:
def remove_repeated_phrases(text):
    # Removes repeated short patterns
    return re.sub(r"(%.?\s*){2,}", "", text)


In [89]:
def remove_garbage_symbols(text):
    # Keep letters, numbers, basic punctuation
    return re.sub(r"[^\x00-\x7F]+", "", text)


In [90]:
def normalize_punctuation(text):
    return re.sub(r"([!?.,])\1{1,}", r"\1", text)


In [91]:
def remove_urls(text):
    return re.sub(r"http\S+|www\.\S+", "", text)


In [92]:
def mask_emails(text):
    return re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", "[EMAIL]", text)


In [93]:
def remove_mentions_hashtags(text):
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    return text


In [94]:
def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()


In [95]:
def remove_unicode_artifacts(text):
    return re.sub(r"[�]", "", text)


In [112]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.encode("utf-8", "ignore").decode("utf-8")
    text = remove_unicode_artifacts(text)
    text = remove_parenthetical_noise(text)
    text = remove_emojis(text)
    text = remove_garbage_symbols(text)
    text = remove_urls(text)
    text = mask_emails(text)
    text = remove_mentions_hashtags(text)
    text = normalize_punctuation(text)
    text = remove_repeated_tokens(text) 
    text = clean_whitespace(text)

    return text



In [113]:
df_talkmap_clean = df_talkmap.copy()
df_talkmap_clean["text"] = df_talkmap_clean["text"].apply(clean_text)

df_talkmap_clean["source"] = "talkmap"
df_talkmap_clean["speaker"] = df_talkmap_clean["speaker"]
df_talkmap_clean = df_talkmap_clean[["source", "speaker", "text"]]


In [114]:
df_comcast_clean = df_comcast.copy()
df_comcast_clean["text"] = df_comcast_clean["Customer Complaint"].apply(clean_text)

df_comcast_clean["source"] = "comcast"
df_comcast_clean["speaker"] = "customer"
df_comcast_clean = df_comcast_clean[["source", "speaker", "text"]]


In [115]:
bitext_user = df_bitext.copy()
bitext_user["text"] = bitext_user["instruction"].apply(clean_text)
bitext_user["source"] = "bitext"
bitext_user["speaker"] = "user"

bitext_agent = df_bitext.copy()
bitext_agent["text"] = bitext_agent["response"].apply(clean_text)
bitext_agent["source"] = "bitext"
bitext_agent["speaker"] = "agent"

df_bitext_clean = pd.concat([
    bitext_user[["source", "speaker", "text"]],
    bitext_agent[["source", "speaker", "text"]]
])

In [100]:
df_talkmap_clean.to_csv(OUT_DIR / "talkmap_clean.csv", index=False)
df_comcast_clean.to_csv(OUT_DIR / "comcast_clean.csv", index=False)
df_bitext_clean.to_csv(OUT_DIR / "bitext_clean.csv", index=False)

In [116]:
df_all = pd.concat([
    df_talkmap_clean,
    df_comcast_clean,
    df_bitext_clean
]).reset_index(drop=True)

df_all = df_all[df_all["text"].str.len() > 10]


In [117]:
df_all.head(-5)

,source,speaker,text
0,talkmap,agent,"You're welcome, Mistie. I apologize again for ..."
1,talkmap,client,That sounds reassuring. But what if someone ha...
2,talkmap,client,"Alright, thank you for your help, Dayna. I app..."
3,talkmap,agent,"Goodbye, Angeline. Have a great day."
4,talkmap,agent,"You're welcome, Lessie. Thank you for choosing..."
...,...,...,...
155958,bitext,agent,"I'm picking up what you're putting down, your ..."
155959,bitext,agent,I'm attuned to the idea that you're seeking as...
155960,bitext,agent,I comprehend your need to stay informed about ...
155961,bitext,agent,I'm conscious of the reality that you have bee...


In [118]:
patterns = {
    "urls": r"http[s]?://|www\.",          # any URL
    "repeated_punct": r"([!?])\1{1,}",     # !! or ??? or more
    "emails": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",
    "numbers": r"\b\d+\b",
    "hashtags": r"#\w+",
    "mentions": r"@\w+",
    "emojis": "[" 
              u"\U0001F600-\U0001F64F"  # emoticons
              u"\U0001F300-\U0001F5FF"  # symbols & pictographs
              u"\U0001F680-\U0001F6FF"  # transport & map symbols
              u"\U0001F1E0-\U0001F1FF"  # flags
              u"\U00002700-\U000027BF"  # dingbats
              u"\U000024C2-\U0001F251"
              "]+"
}


In [119]:
def filter_by_pattern(df, column, pattern_name):
    regex = patterns[pattern_name]
    filtered = df[df[column].str.contains(regex, na=False)]
    return filtered

def show_examples(df, pattern_name, n=5):
    filtered = filter_by_pattern(df, TEXT_COL, pattern_name)
    print(f"--- {pattern_name} ({len(filtered)} messages) ---")
    display(filtered[[TEXT_COL]].head(n))


In [120]:
TEXT_COL = "text"

In [121]:
for name in patterns:
    show_examples(df_all, name, n=5)

--- urls (0 messages) ---


,text


C:\Users\nourg\AppData\Local\Temp\ipykernel_1396\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


--- repeated_punct (0 messages) ---


,text


...

,text


--- mentions (0 messages) ---


,text


--- emojis (0 messages) ---


,text


In [122]:
counts = {name: len(filter_by_pattern(df_all, TEXT_COL, name)) for name in patterns}
pd.DataFrame(list(counts.items()), columns=["Pattern", "Message Count"])

C:\Users\nourg\AppData\Local\Temp\ipykernel_1396\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,Pattern,Message Count
0,urls,0
1,repeated_punct,0
2,emails,0
3,numbers,14745
4,hashtags,0
5,mentions,0
6,emojis,0


In [126]:
df_all.to_csv(OUT_DIR / "all_clean_for_ner.csv", index=False)